# 🏦 AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  
**Stage 3:** Machine Learning Risk Scoring

---
So far:
- **Stage 1** → Built a dataset of 500 transactions
- **Stage 2** → Applied 7 rule-based red flags

The problem with rules alone is that they are **rigid** — they only catch what you already know to look for.  
What about suspicious transactions that don't match any known rule?

That's where **Machine Learning** comes in.

In Stage 3, we train a model that learns what a *normal* transaction looks like — and then flags anything that looks *abnormal*, even if it doesn't match a specific rule.

The output will be a **Risk Score from 0 to 100** for every transaction.  
Higher score = more suspicious.

---
## Step 1: Re-run Stages 1 & 2
We need our dataset with red flags ready. This is the combined code from Stage 1 and Stage 2.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']
countries = ['USA', 'UK', 'India', 'Cayman Islands', 'Panama',
             'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore']
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']
transaction_types = ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
                     'Online Transfer', 'Check', 'Crypto Exchange']
n = 500

df = pd.DataFrame({
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],
    'Customer_Type': np.random.choice(customer_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),
            np.random.exponential(scale=3000, size=n).clip(100, 100000)
        ), 2),
    'Transaction_Type': np.random.choice(transaction_types, n),
    'Origin_Country': np.random.choice(countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]),
    'Destination_Country': np.random.choice(countries, n),
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2),
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

def apply_red_flags(row):
    flags = []
    if 9000 <= row['Transaction_Amount'] <= 9999:
        flags.append('Structuring')
    if row['Origin_Country'] in high_risk_countries or \
       row['Destination_Country'] in high_risk_countries:
        flags.append('High-Risk Country')
    if row['Avg_Transaction_Last_6Months'] > 0:
        if row['Transaction_Amount'] / row['Avg_Transaction_Last_6Months'] > 5:
            flags.append('Unusual Amount vs History')
    if row['Customer_Type'] in ['Shell Company', 'NGO'] and \
       row['Transaction_Type'] == 'Wire Transfer':
        flags.append('High-Risk Entity Type')
    if row['Prior_SAR_Filed'] == 1:
        flags.append('Prior SAR on File')
    if row['Account_Age_Years'] < 1 and row['Transaction_Amount'] > 10000:
        flags.append('New Account High Value')
    if row['Num_Transactions_Last_30Days'] > 30:
        flags.append('Excessive Transaction Frequency')
    return '; '.join(flags) if flags else 'None'

df['Red_Flags'] = df.apply(apply_red_flags, axis=1)
df['Flag_Count'] = df['Red_Flags'].apply(
    lambda x: 0 if x == 'None' else len(x.split(';')))

print(f'✅ Stages 1 & 2 complete — {len(df)} transactions with red flags ready.')

---
## Step 2: Import the ML Libraries

We need two new tools from scikit-learn (a Python ML library):

- **IsolationForest** — the algorithm that detects anomalies
- **StandardScaler** — a tool to normalize our data before feeding it to the model

> **What is normalization?**  
> Our features have very different scales. Transaction Amount can be $100,000 while Account Age is 5 years. If we feed these raw numbers to the model, the large numbers will dominate. Normalization rescales everything to the same range so each feature gets a fair weight.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print('✅ ML libraries loaded!')

---
## Step 3: Choose the Features

**Features** are the columns we feed into the model. Not every column is useful.

We can't use text columns like `Transaction_ID` or `Customer_Type` directly — the model only understands numbers.

We select 6 numeric columns that capture different dimensions of risk:

| Feature | Why it matters |
|---|---|
| Transaction_Amount | Large or unusual amounts are suspicious |
| Num_Transactions_Last_30Days | High frequency = possible smurfing |
| Avg_Transaction_Last_6Months | Helps detect deviation from normal behavior |
| Account_Age_Years | New accounts are higher risk |
| Prior_SAR_Filed | Already flagged by compliance before |
| Flag_Count | How many rules fired on this transaction |

In [ ]:
features = [
    'Transaction_Amount',
    'Num_Transactions_Last_30Days',
    'Avg_Transaction_Last_6Months',
    'Account_Age_Years',
    'Prior_SAR_Filed',
    'Flag_Count'
]

# Extract just these columns into a new variable X
# X is the standard name for input data in ML
X = df[features]

print('✅ Features selected!')
print(f'   Shape of X: {X.shape}  (500 rows, 6 features)')
print('\n📋 Preview of feature data:')
X.head()

---
## Step 4: Normalize the Features

`StandardScaler` transforms each column so it has:
- **Mean of 0** (average value becomes 0)
- **Standard deviation of 1** (values spread equally)

After scaling, a Transaction_Amount of \$50,000 and an Account_Age of 15 years are both expressed on the same scale — so neither one unfairly dominates the model.

`fit_transform()` does two things at once:
1. **fit** — learns the mean and standard deviation of each column
2. **transform** — applies the scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✅ Features normalized!')
print(f'   Shape after scaling: {X_scaled.shape}')
print(f'\n   Example — Transaction_Amount before scaling: ${df["Transaction_Amount"].iloc[0]:,.2f}')
print(f'   Example — Transaction_Amount after scaling : {X_scaled[0][0]:.4f}')

---
## Step 5: Train the Isolation Forest Model

### How does Isolation Forest work?

Imagine you have a forest of decision trees. Each tree randomly picks a feature and a split point to *isolate* one transaction from the rest.

- **Normal transactions** are similar to many others → they take **many splits** to isolate
- **Anomalous transactions** are unusual → they get isolated in **very few splits**

The fewer splits needed to isolate a transaction, the more suspicious it is.

Think of it like this: if you're trying to find the one person in a crowd who is acting differently, you can isolate them quickly. Finding a normal person takes longer because they blend in.

> **Why Isolation Forest for AML?**  
> In real AML, we almost never have labeled data (we don't know which transactions are truly fraudulent). Isolation Forest is **unsupervised** — it doesn't need labels. It just learns what normal looks like and flags deviations.

### Key parameter: `contamination`
This tells the model what percentage of transactions we expect to be anomalous.  
We set it to **0.08 (8%)** — a reasonable estimate for AML alert rates in banking.

In [ ]:
# Initialize the model
model = IsolationForest(
    contamination=0.08,   # We expect ~8% of transactions to be anomalous
    random_state=42       # For reproducibility
)

# Train the model on our scaled data
# This is where the model learns what 'normal' looks like
model.fit(X_scaled)

print('✅ Isolation Forest model trained!')

---
## Step 6: Generate Risk Scores

`score_samples()` returns a raw anomaly score for each transaction.

- Scores are **negative numbers** — closer to 0 means more normal, more negative means more anomalous
- Example: -0.1 = fairly normal, -0.6 = very suspicious

These negative numbers are not intuitive, so we convert them to a **0–100 scale** where **100 = most suspicious**.

In [ ]:
# Get raw anomaly scores from the model
raw_scores = model.score_samples(X_scaled)

print('Raw score examples (more negative = more suspicious):')
print(f'   Most normal score  : {raw_scores.max():.4f}')
print(f'   Most anomalous score: {raw_scores.min():.4f}')

# Convert to 0-100 scale
# Formula: flip and normalize so highest risk = 100
min_score = raw_scores.min()
max_score = raw_scores.max()

df['Risk_Score'] = ((raw_scores - max_score) / (min_score - max_score) * 100).round(1)

print(f'\n✅ Risk scores calculated!')
print(f'   Score range: {df["Risk_Score"].min()} to {df["Risk_Score"].max()}')
print(f'   Average score: {df["Risk_Score"].mean():.1f}')

---
## Step 7: Assign Risk Tiers

A raw score is useful, but compliance teams work with **risk tiers** — simple categories that tell them what action to take.

| Tier | Score | Action |
|---|---|---|
| HIGH | ≥ 70 | Escalate immediately — consider SAR filing |
| MEDIUM | 40–69 | Enhanced Due Diligence (EDD) required |
| LOW | < 40 | Standard monitoring, no immediate action |

In [ ]:
def assign_tier(score):
    if score >= 70:
        return 'HIGH'
    elif score >= 40:
        return 'MEDIUM'
    else:
        return 'LOW'

df['Risk_Tier'] = df['Risk_Score'].apply(assign_tier)

# Mark transactions that need immediate attention
df['Alert'] = df['Risk_Tier'].apply(lambda x: 'YES' if x == 'HIGH' else 'NO')

print('✅ Risk tiers assigned!')
print('\n📊 Risk Tier Breakdown:')
tier_counts = df['Risk_Tier'].value_counts()
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    count = tier_counts.get(tier, 0)
    pct = round(count / len(df) * 100, 1)
    print(f'   {tier:<8} : {count:>4} transactions ({pct}%)')

---
## Step 8: Analyse the Results

In [ ]:
# Show the top 10 highest risk transactions
print('🔴 Top 10 Highest Risk Transactions:')
df.sort_values('Risk_Score', ascending=False)[
    ['Transaction_ID', 'Customer_Type', 'Transaction_Amount',
     'Origin_Country', 'Red_Flags', 'Risk_Score', 'Risk_Tier']
].head(10)

In [ ]:
# Compare average risk score by customer type
print('👤 Average Risk Score by Customer Type:')
df.groupby('Customer_Type')['Risk_Score'].mean().round(1).sort_values(ascending=False)

In [ ]:
# Compare average risk score by transaction type
print('💳 Average Risk Score by Transaction Type:')
df.groupby('Transaction_Type')['Risk_Score'].mean().round(1).sort_values(ascending=False)

In [ ]:
# Do flagged transactions actually get higher scores? They should!
print('🔍 Does the ML model agree with our rules?')
print('\nAverage Risk Score:')
print(f'   Transactions WITH red flags : {df[df["Flag_Count"] > 0]["Risk_Score"].mean():.1f}')
print(f'   Transactions WITHOUT flags  : {df[df["Flag_Count"] == 0]["Risk_Score"].mean():.1f}')
print('\nIf the ML score is higher for flagged transactions, our model is working correctly!')

---
## ✅ Stage 3 Complete!

You now have a **risk score and risk tier** for every transaction.

### What you just built in real-world terms:
- Rules alone (Stage 2) = a basic compliance checklist
- Rules + ML scoring (Stage 3) = a modern transaction monitoring system like the ones used at JPMorgan, HSBC, and Citi
- The ML layer catches suspicious transactions that don't match any known rule — this is what makes it powerful

---
### Before moving to Stage 4, answer these:

1. **What is the difference between supervised and unsupervised ML?  Why did we use unsupervised here?**
2. **Why did we normalize the features before training the model?**
3. **Look at your output — is the average risk score higher for flagged transactions than unflagged ones? What does that tell you?**
4. **What does the `contamination` parameter mean in plain English?**

---
**Next → Stage 4: Excel Report**  
We export everything into a professional, formatted Excel report with multiple sheets — the kind you'd actually hand to a senior compliance officer.